# DeepSeek-V4-Mini Lecture 2: Attention V4

1. HCA
2. SWA
3. CSA

## 0. Config & Data

In [12]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
torch.manual_seed(42)

In [13]:
# config 
@dataclass
class AttentionModelArgs:
    vocab_size: int = 100
    dim: int = 512
    head_dim: int = 64
    n_heads: int = 8
    n_kv_heads: int = 8
    window_size: int = 128
    compress_ratios: int = 4 # 0, 4, 12
    attn_top_k: int = 11
    
args = AttentionModelArgs()

# data
bsz = 2
seq_len = 8192
X = torch.randn(bsz, seq_len, args.dim)

## 1. HCA

### 1.1 Compressor

In [14]:
class Compressor(nn.Module):
    def __init__(self, compress_ratios, dim):
        super().__init__()
        self.compress_ratios = compress_ratios
        self.w_compress = nn.Linear(compress_ratios, 1, bias=False)
        
    def forward(self, X):
        '''
            output: [B, n_c, D]
        '''
        B, L, D = X.shape
        c, n_c = self.compress_ratios, L // self.compress_ratios
        if self.compress_ratios > 1:
            X_ = X.reshape(B, n_c, c, D) 
            X_ = self.w_compress(X_.transpose(2,3))[...,0] # [B,n_c, D, C]  [C,1]
        return X_

### 1.2 Overlap

In [15]:
def overlap(X):
    B,L,D = X.shape
    X_ = torch.zeros(B, L, D*2)
    X_[:, 1:, :D] = X[:, :L-1, :]
    X_[:, :-1, D:] = X[:, 1:, :]
    return X_

### 1.3 HCA

In [16]:
Q = torch.randn(bsz, seq_len, args.dim)
K = torch.randn(bsz, seq_len, args.dim//2)
V = torch.randn(bsz, seq_len, args.dim//2)

compressor = Compressor(args.compress_ratios, args.dim//2)
K_ = overlap(compressor(K))
V_ = overlap(compressor(V))

# do Q,K,V, Attntion
def attn(Q,K,V):
    return Q@K.transpose(-2,-1)@V
    
O_attn = attn(Q, K_, V_) # is not right

### 1.4 HCA with Casual Compress Key/Value

In [17]:
O = torch.zeros_like(Q)
for i in range(bsz):
    for j in range(seq_len):
        j_ = j // args.compress_ratios + 1
        K_c, V_c = K_[i, :j_], V_[i, :j_]
        O[i,j] = Q[i, [j]] @ K_c.transpose(-2,-1) @ V_c

## 2. HCA With SWA

HCA will skip last KV-block, need SWA get closer-position original KV

In [22]:
win = 128
O = torch.zeros_like(Q)
for i in range(bsz):
    for j in range(seq_len):
        # HCA
        j_ = j // args.compress_ratios + 1
        K_c, V_c = K_[i, :j_], V_[i, :j_]
        print(K_c.transpose(-2,-1) @ V_c.shape)
        O[i,j,:] += Q[i, [j]] @ K_c.transpose(-2,-1) @ V_c

        # SWA
        # left, right = max(0, j-win), min(seq_len, j+win)
        # K_s, V_s = K[i, left:right], V[i, left:right]
        # O[i,j] += Q[i, [j]] @ K_s.transpose(-2,-1) @ V_s

TypeError: unsupported operand type(s) for @: 'Tensor' and 'torch.Size'

## 3. CSA

### 3.1 Indexer

In [19]:
class Indexer(nn.Module):
    def __init__(self, args):
        super().__init__()
        dim, n_head = args.dim, args.n_heads
        self.Wk_light = nn.Linear(dim, 1, bias = False)
        self.Wq_light = nn.Linear(dim, 1, bias = False)
        
    def forward(self, Q, K, topk):
        Q_, K_ = self.Wq_light(Q), self.Wk_light(K)
        S = Q_ @ K_.transpose(-2, -1)
        _, idx = torch.topk(S, topk, dim = -1)
        return idx

### 1.2 CSA

In [ ]:
Q = torch.randn(bsz, seq_len, args.dim)
K = torch.randn(bsz, seq_len, args.dim)
V = torch.randn(bsz, seq_len, args.dim)

# for CSA compress
args.compress_ratios = 4
compressor = Compressor(args.compress_ratios, args.dim)
K_ = compressor(K)
V_ = compressor(V)

# for CSA pre top-k 
indexer = Indexer(args)
idx = indexer(Q, K_, args.attn_top_k)

# do CSA Attn
z = torch.zeros_like(Q)
for i in range(bsz):
    for j in range(seq_len):
        sparse_ids = idx[i, j]
        z[i,j] = attn(Q[i, [j]], K_[i, sparse_ids], V_[i, sparse_ids])

### 3.2 CSA With SWA

In [ ]:
win = 128

# do CSA with SWA
z = torch.zeros_like(Q)
for i in range(bsz):
    for j in range(seq_len):
        sparse_ids = idx[i, j]
        left, right = max(0, j-win), min(seq_len, j+win)

        K_mix = torch.cat((K_[i, sparse_ids], K[i, left:right]), dim = 0) 
        V_mix = torch.cat((V_[i, sparse_ids], V[i, left:right]), dim = 0)


        z[i,j] = attn(Q[i, [j]], K_mix, V_mix)